In [1]:
import subprocess
import sys

packages = [
    "torch-geometric",
    "pandas",
    "numpy",
    "scikit-learn",
    "tqdm",
]

print("Installing packages...")
for pkg in packages:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", pkg])

import torch
import torch_geometric
print(f"PyTorch  : {torch.__version__}")
print(f"PyG      : {torch_geometric.__version__}")
print(f"CUDA     : {torch.cuda.is_available()}")
print(f"pyg-lib  : {torch_geometric.typing.WITH_PYG_LIB}")
print(f"torch-sparse: {torch_geometric.typing.WITH_TORCH_SPARSE}")

Installing packages...
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 63.7/63.7 kB 2.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.3/1.3 MB 23.4 MB/s eta 0:00:00
PyTorch  : 2.10.0+cu128
PyG      : 2.7.0
CUDA     : True
pyg-lib  : False
torch-sparse: False


In [2]:
import os
import json
import time
import warnings
from collections import defaultdict, deque
from datetime import datetime

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from sklearn.metrics import f1_score, precision_score, recall_score, roc_auc_score
from sklearn.preprocessing import LabelEncoder
from torch_geometric.data import Data
from torch_geometric.nn import GATConv
from tqdm.notebook import trange, tqdm

warnings.filterwarnings("ignore")

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {DEVICE}")

Device: cuda


In [4]:
class Config:
    """
    Change DATASET_NAME to switch between datasets.
    Supported values (case-sensitive, must match filename prefix):
        LI-Small, HI-Small, LI-Medium, HI-Medium

    Colab free T4 memory guide:
        LI-Small  ~ 7M  transactions  -> fits comfortably
        HI-Small  ~ 5M  transactions  -> fits comfortably
        LI-Medium ~ 31M transactions  -> marginal, may OOM
        HI-Medium ~ 32M transactions  -> marginal, may OOM

    Recommended for free Colab: LI-Small or HI-Small
    """

    DATASET_NAME = "LI-Small"   # <-- change only this line to swap datasets

    # Colab path after mounting Kaggle or uploading manually
    DATA_PATH = "/kaggle/input/datasets/ealtman2019/ibm-transactions-for-anti-money-laundering-aml/"

    # Model
    HIDDEN_DIM    = 64
    NUM_LAYERS    = 2
    NUM_HEADS     = 4       # GAT attention heads
    DROPOUT       = 0.3

    # Training
    EPOCHS                   = 50
    LEARNING_RATE            = 0.001
    WEIGHT_DECAY             = 1e-5
    MINORITY_CLASS_WEIGHT    = 10.0   # upweight laundering class
    EARLY_STOPPING_PATIENCE  = 10

    # Inference
    SUSPICION_THRESHOLD = 0.5   # probability threshold for flagging

    DEVICE = DEVICE

cfg = Config()
print(f"Dataset : {cfg.DATASET_NAME}")
print(f"Path    : {cfg.DATA_PATH}")
print(f"Device  : {cfg.DEVICE}")

Dataset : LI-Small
Path    : /kaggle/input/datasets/ealtman2019/ibm-transactions-for-anti-money-laundering-aml/
Device  : cuda


In [ ]:
# Run this cell only if you are uploading files manually.
# If you use Kaggle API or have files already at DATA_PATH, skip this cell.

from google.colab import files

os.makedirs(cfg.DATA_PATH, exist_ok=True)
print(f"Upload the following three files for dataset '{cfg.DATASET_NAME}':")
print(f"  {cfg.DATASET_NAME}_Trans.csv")
print(f"  {cfg.DATASET_NAME}_accounts.csv")
print(f"  {cfg.DATASET_NAME}_Patterns.txt")

uploaded = files.upload()
for filename in uploaded.keys():
    dst = os.path.join(cfg.DATA_PATH, filename)
    with open(dst, "wb") as f:
        f.write(uploaded[filename])
    print(f"Saved: {dst}")

In [5]:
def load_dataset(cfg):
    base = os.path.join(cfg.DATA_PATH, cfg.DATASET_NAME)
    print(f"\nLoading {cfg.DATASET_NAME} ...")

    trans_df    = pd.read_csv(f"{base}_Trans.csv")
    accounts_df = pd.read_csv(f"{base}_accounts.csv")

    patterns_path = f"{base}_Patterns.txt"
    patterns_text = open(patterns_path).read() if os.path.exists(patterns_path) else ""

    print(f"  Transactions : {trans_df.shape}")
    print(f"  Accounts     : {accounts_df.shape}")
    print(f"  Patterns file: {'found' if patterns_text else 'not found'}")
    return trans_df, accounts_df, patterns_text

trans_df, accounts_df, patterns_text = load_dataset(cfg)
print("\nTransaction columns:")
print(list(trans_df.columns))
print("\nAccount columns:")
print(list(accounts_df.columns))
print("\nSample transactions:")
trans_df.head(3)


Loading LI-Small ...
  Transactions : (6924049, 11)
  Accounts     : (712688, 5)
  Patterns file: found

Transaction columns:
['Timestamp', 'From Bank', 'Account', 'To Bank', 'Account.1', 'Amount Received', 'Receiving Currency', 'Amount Paid', 'Payment Currency', 'Payment Format', 'Is Laundering']

Account columns:
['Bank Name', 'Bank ID', 'Account Number', 'Entity ID', 'Entity Name']

Sample transactions:


,Timestamp,From Bank,Account,To Bank,Account.1,Amount Received,Receiving Currency,Amount Paid,Payment Currency,Payment Format,Is Laundering
0,2022/09/01 00:08,11,8000ECA90,11,8000ECA90,3195403.00,US Dollar,3195403.00,US Dollar,Reinvestment,0
1,2022/09/01 00:21,3402,80021DAD0,3402,80021DAD0,1858.96,US Dollar,1858.96,US Dollar,Reinvestment,0
2,2022/09/01 00:00,11,8000ECA90,1120,8006AA910,592571.00,US Dollar,592571.00,US Dollar,Cheque,0


In [6]:
def preprocess(trans_df, accounts_df):
    """
    Column schema
    -------------
    Trans   : Timestamp, From Bank, Account, To Bank, Account.1,
              Amount Received, Receiving Currency, Amount Paid,
              Payment Currency, Payment Format, Is Laundering
    Accounts: Bank ID, Account Number, Entity ID, Entity Name
    """
    print("Preprocessing ...")

    # ---- account id = "BankID_AccountNumber" ----
    acc_col   = accounts_df.columns[1]   # Account Number
    bank_col  = accounts_df.columns[0]   # Bank ID
    accounts_df["acc_id"] = (
        accounts_df[bank_col].astype(str) + "_" +
        accounts_df[acc_col].astype(str)
    )

    df = trans_df.copy()

    # Build consistent account IDs from transaction table
    df["From_Account"] = df["From Bank"].astype(str) + "_" + df["Account"].astype(str)
    df["To_Account"]   = df["To Bank"].astype(str)   + "_" + df["Account.1"].astype(str)

    # Timestamp
    df["Timestamp"] = pd.to_datetime(df["Timestamp"], errors="coerce")
    df = df.dropna(subset=["From_Account", "To_Account", "Timestamp"])

    # Label
    if "Is Laundering" in df.columns:
        df["label"] = df["Is Laundering"].astype(int)
    else:
        df["label"] = 0

    # Payment format encoding
    le = LabelEncoder()
    df["payment_enc"] = le.fit_transform(df["Payment Format"].astype(str))

    # Currency encoding
    le2 = LabelEncoder()
    df["currency_enc"] = le2.fit_transform(df["Payment Currency"].astype(str))

    # Numeric features for edges
    df["amount_log"]      = np.log1p(df["Amount Paid"].fillna(0))
    df["hour"]            = df["Timestamp"].dt.hour / 23.0
    df["day_of_week"]     = df["Timestamp"].dt.dayofweek / 6.0
    df["unix_ts"]         = df["Timestamp"].astype(np.int64) / 1e18   # normalised

    df = df.reset_index(drop=True)

    print(f"  Cleaned transactions : {len(df)}")
    print(f"  Laundering rate      : {df['label'].mean():.5f}")

    return df, le, le2

trans_clean, le_payment, le_currency = preprocess(trans_df, accounts_df)

Preprocessing ...
  Cleaned transactions : 6924049
  Laundering rate      : 0.00051


In [7]:
def build_graph(df):
    """
    Node  = unique bank account
    Edge  = transaction (directed, From -> To)
    Edge features: [amount_log, hour, day_of_week, payment_enc_norm, currency_enc_norm]
    Node features: aggregated statistics over incident transactions
    Labels are on edges.
    
    Because full-graph training is used (no LinkNeighborLoader),
    this works without pyg-lib or torch-sparse.
    """
    print("Building graph ...")

    # ---- node index mapping ----
    all_accounts = pd.unique(
        pd.concat([df["From_Account"], df["To_Account"]])
    )
    node2idx = {acc: i for i, acc in enumerate(all_accounts)}
    idx2node = {i: acc for acc, i in node2idx.items()}
    num_nodes = len(node2idx)

    src = df["From_Account"].map(node2idx).values.astype(np.int64)
    dst = df["To_Account"].map(node2idx).values.astype(np.int64)

    edge_index = torch.tensor(np.stack([src, dst], axis=0), dtype=torch.long)

    # ---- edge features ----
    num_pay  = df["payment_enc"].max() + 1
    num_cur  = df["currency_enc"].max() + 1

    edge_feat = np.stack([
        df["amount_log"].values,
        df["hour"].values,
        df["day_of_week"].values,
        df["payment_enc"].values / max(num_pay, 1),
        df["currency_enc"].values / max(num_cur, 1),
    ], axis=1).astype(np.float32)

    edge_attr = torch.tensor(edge_feat, dtype=torch.float)
    edge_y    = torch.tensor(df["label"].values, dtype=torch.long)

    # ---- node features: per-node stats (out-degree, in-degree, mean sent, mean received) ----
    node_feat = np.zeros((num_nodes, 4), dtype=np.float32)
    amounts   = df["amount_log"].values

    np.add.at(node_feat[:, 0], src, 1)             # out-degree count
    np.add.at(node_feat[:, 1], dst, 1)             # in-degree count
    np.add.at(node_feat[:, 2], src, amounts)       # sum sent
    np.add.at(node_feat[:, 3], dst, amounts)       # sum received

    # Normalise
    max_vals = node_feat.max(axis=0, keepdims=True) + 1e-8
    node_feat = node_feat / max_vals

    x = torch.tensor(node_feat, dtype=torch.float)

    data = Data(x=x, edge_index=edge_index, edge_attr=edge_attr, edge_y=edge_y)
    data.num_nodes = num_nodes
    data.node2idx  = node2idx
    data.idx2node  = idx2node

    print(f"  Nodes        : {num_nodes:,}")
    print(f"  Edges        : {edge_index.shape[1]:,}")
    print(f"  Laundering   : {edge_y.sum().item():,}  ({edge_y.float().mean().item():.5f})")
    print(f"  Node feat dim: {x.shape[1]}")
    print(f"  Edge feat dim: {edge_attr.shape[1]}")
    return data

graph_data = build_graph(trans_clean)

Building graph ...
  Nodes        : 705,907
  Edges        : 6,924,049
  Laundering   : 3,565  (0.00051)
  Node feat dim: 4
  Edge feat dim: 5


In [8]:
def temporal_split(df, graph_data, train_frac=0.6, val_frac=0.2):
    """
    60 / 20 / 20 temporal split matching the paper's methodology.
    Splits on edge indices ordered by Timestamp.
    """
    n = len(df)
    sorted_idx = df["Timestamp"].argsort().values   # ascending time order

    t1 = int(n * train_frac)
    t2 = int(n * (train_frac + val_frac))

    train_idx = sorted_idx[:t1]
    val_idx   = sorted_idx[t1:t2]
    test_idx  = sorted_idx[t2:]

    def make_mask(indices):
        mask = torch.zeros(n, dtype=torch.bool)
        mask[indices] = True
        return mask

    graph_data.train_mask = make_mask(train_idx)
    graph_data.val_mask   = make_mask(val_idx)
    graph_data.test_mask  = make_mask(test_idx)

    print(f"Split  ->  train: {train_idx.shape[0]:,}  |  val: {val_idx.shape[0]:,}  |  test: {test_idx.shape[0]:,}")
    return graph_data

graph_data = temporal_split(trans_clean, graph_data)

Split  ->  train: 4,154,429  |  val: 1,384,810  |  test: 1,384,810


In [9]:
class EdgeGAT(nn.Module):
    """
    Graph Attention Network for edge (transaction) classification.

    Architecture
    ------------
    1. Two GAT layers produce node embeddings using node features.
    2. Each edge embedding is formed by concatenating:
         [src_node_emb || dst_node_emb || edge_attr]
    3. An MLP classifies each edge as laundering / normal.

    This architecture matches the GIN+EU approach referenced in the paper
    (edge updates on top of node message passing) and avoids LinkNeighborLoader,
    which requires pyg-lib or torch-sparse.
    """

    def __init__(self, node_feat_dim, edge_feat_dim, hidden_dim, num_classes,
                 num_layers=2, num_heads=4, dropout=0.3):
        super().__init__()

        self.gat_layers = nn.ModuleList()
        self.norms      = nn.ModuleList()

        in_dim = node_feat_dim
        for i in range(num_layers):
            out_dim  = hidden_dim // num_heads
            is_last  = (i == num_layers - 1)
            heads    = 1 if is_last else num_heads
            concat   = not is_last
            self.gat_layers.append(
                GATConv(in_dim, out_dim, heads=heads, concat=concat,
                        dropout=dropout, add_self_loops=True)
            )
            self.norms.append(nn.BatchNorm1d(out_dim * heads if concat else out_dim))
            in_dim = out_dim * heads if concat else out_dim

        node_emb_dim = in_dim

        # Edge MLP: src_emb + dst_emb + edge_feat
        mlp_in = node_emb_dim * 2 + edge_feat_dim
        self.edge_mlp = nn.Sequential(
            nn.Linear(mlp_in, hidden_dim),
            nn.BatchNorm1d(hidden_dim),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(hidden_dim, hidden_dim // 2),
            nn.ReLU(),
            nn.Linear(hidden_dim // 2, num_classes),
        )

        self.dropout = dropout

    def get_node_embeddings(self, x, edge_index):
        for gat, norm in zip(self.gat_layers, self.norms):
            x = gat(x, edge_index)
            x = norm(x)
            x = F.elu(x)
            x = F.dropout(x, p=self.dropout, training=self.training)
        return x

    def forward(self, x, edge_index, edge_attr):
        node_emb = self.get_node_embeddings(x, edge_index)

        src_idx = edge_index[0]
        dst_idx = edge_index[1]

        edge_emb = torch.cat([node_emb[src_idx], node_emb[dst_idx], edge_attr], dim=-1)
        logits   = self.edge_mlp(edge_emb)
        return logits, node_emb


# Instantiate
model = EdgeGAT(
    node_feat_dim = graph_data.x.shape[1],
    edge_feat_dim = graph_data.edge_attr.shape[1],
    hidden_dim    = cfg.HIDDEN_DIM,
    num_classes   = 2,
    num_layers    = cfg.NUM_LAYERS,
    num_heads     = cfg.NUM_HEADS,
    dropout       = cfg.DROPOUT,
).to(cfg.DEVICE)

total_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(model)
print(f"\nTrainable parameters: {total_params:,}")

EdgeGAT(
  (gat_layers): ModuleList(
    (0): GATConv(4, 16, heads=4)
    (1): GATConv(64, 16, heads=1)
  )
  (norms): ModuleList(
    (0): BatchNorm1d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (1): BatchNorm1d(16, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
  )
  (edge_mlp): Sequential(
    (0): Linear(in_features=37, out_features=64, bias=True)
    (1): BatchNorm1d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (2): ReLU()
    (3): Dropout(p=0.3, inplace=False)
    (4): Linear(in_features=64, out_features=32, bias=True)
    (5): ReLU()
    (6): Linear(in_features=32, out_features=2, bias=True)
  )
)

Trainable parameters: 6,386


In [11]:
def compute_class_weight(edge_y):
    n_neg = (edge_y == 0).sum().item()
    n_pos = (edge_y == 1).sum().item()
    total = n_neg + n_pos
    # inverse frequency
    w_neg = total / (2 * n_neg + 1e-8)
    w_pos = total / (2 * n_pos + 1e-8)
    # additional manual upweight for minority
    w_pos *= cfg.MINORITY_CLASS_WEIGHT
    return torch.tensor([w_neg, w_pos], dtype=torch.float)


class_weights = compute_class_weight(graph_data.edge_y).to(cfg.DEVICE)
criterion = nn.CrossEntropyLoss(weight=class_weights)
optimizer = torch.optim.Adam(model.parameters(), lr=cfg.LEARNING_RATE,
                             weight_decay=cfg.WEIGHT_DECAY)
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
    optimizer, mode="max", factor=0.5, patience=5
)

print(f"Class weights: neg={class_weights[0]:.3f}  pos={class_weights[1]:.3f}")


def move_graph_to_device(data, device):
    data.x          = data.x.to(device)
    data.edge_index = data.edge_index.to(device)
    data.edge_attr  = data.edge_attr.to(device)
    data.edge_y     = data.edge_y.to(device)
    return data


def train_epoch(model, data, mask, optimizer, criterion, device):
    model.train()
    optimizer.zero_grad()
    logits, _ = model(data.x, data.edge_index, data.edge_attr)
    loss = criterion(logits[mask], data.edge_y[mask])
    loss.backward()
    torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
    optimizer.step()
    return loss.item()


@torch.no_grad()
def evaluate(model, data, mask, device):
    model.eval()
    logits, _ = model(data.x, data.edge_index, data.edge_attr)
    loss   = criterion(logits[mask], data.edge_y[mask]).item()
    probs  = F.softmax(logits[mask], dim=-1)[:, 1].cpu().numpy()
    labels = data.edge_y[mask].cpu().numpy()
    preds  = (probs >= cfg.SUSPICION_THRESHOLD).astype(int)
    f1     = f1_score(labels, preds, zero_division=0)
    return loss, f1, preds, probs

Class weights: neg=0.500  pos=9711.148


In [18]:
# Move full graph to GPU once; full-graph forward pass — no DataLoader needed,
# so no pyg-lib / torch-sparse dependency.
graph_data = move_graph_to_device(graph_data, cfg.DEVICE)

history = {"train_loss": [], "val_loss": [], "val_f1": []}
best_val_f1   = 0.0
patience_ctr  = 0
best_state    = None
save_path = os.path.join("/kaggle/working/", f"model_{cfg.DATASET_NAME}.pth")
print(f"Starting full-graph training on {cfg.DEVICE}  ({cfg.EPOCHS} epochs)\n")

for epoch in trange(cfg.EPOCHS, desc="Training"):
    train_loss = train_epoch(
        model, graph_data, graph_data.train_mask, optimizer, criterion, cfg.DEVICE
    )
    val_loss, val_f1, _, _ = evaluate(
        model, graph_data, graph_data.val_mask, cfg.DEVICE
    )

    history["train_loss"].append(train_loss)
    history["val_loss"].append(val_loss)
    history["val_f1"].append(val_f1)

    scheduler.step(val_f1)

    if val_f1 > best_val_f1:
        best_val_f1  = val_f1
        best_state   = {k: v.clone() for k, v in model.state_dict().items()}
        patience_ctr = 0
    else:
        patience_ctr += 1

    if (epoch + 1) % 5 == 0:
        print(f"  Epoch {epoch+1:3d} | train_loss={train_loss:.4f} | "
              f"val_loss={val_loss:.4f} | val_f1={val_f1:.4f} | "
              f"best_val_f1={best_val_f1:.4f}")

    if patience_ctr >= cfg.EARLY_STOPPING_PATIENCE:
        print(f"\nEarly stopping at epoch {epoch+1}")
        break

# Restore best checkpoint
if best_state is not None:
    model.load_state_dict(best_state,save_path)

print(f"\nBest Validation F1: {best_val_f1:.4f}")

Starting full-graph training on cuda  (50 epochs)



Training:   0%|          | 0/50 [00:00<?, ?it/s]

  Epoch   5 | train_loss=0.6086 | val_loss=0.5852 | val_f1=0.0012 | best_val_f1=0.0012
  Epoch  10 | train_loss=0.5923 | val_loss=0.5765 | val_f1=0.0012 | best_val_f1=0.0012

Early stopping at epoch 11

Best Validation F1: 0.0012


In [20]:
test_loss, test_f1, test_preds, test_probs = evaluate(
    model, graph_data, graph_data.test_mask, cfg.DEVICE
)
test_labels = graph_data.edge_y[graph_data.test_mask].cpu().numpy()

precision = precision_score(test_labels, test_preds, zero_division=0)
recall    = recall_score(test_labels, test_preds, zero_division=0)
try:
    auc = roc_auc_score(test_labels, test_probs)
except ValueError:
    auc = float("nan")

print("=" * 50)
print("TEST SET RESULTS")
print("=" * 50)
print(f"  F1 Score  : {test_f1:.4f}")
print(f"  Precision : {precision:.4f}")
print(f"  Recall    : {recall:.4f}")
print(f"  ROC-AUC   : {auc:.4f}")
print(f"  Loss      : {test_loss:.4f}")
print("=" * 50)

TEST SET RESULTS
  F1 Score  : 0.0013
  Precision : 0.0007
  Recall    : 1.0000
  ROC-AUC   : 0.7505
  Loss      : 0.6005


In [ ]:
@torch.no_grad()
def get_all_probs(model, data, device):
    model.eval()
    logits, node_emb = model(data.x, data.edge_index, data.edge_attr)
    probs = F.softmax(logits, dim=-1)[:, 1].cpu().numpy()
    return probs, node_emb.cpu().numpy()

all_probs, node_emb = get_all_probs(model, graph_data, cfg.DEVICE)

# Flag suspicious edges
suspicious_mask_np = all_probs >= cfg.SUSPICION_THRESHOLD
suspicious_indices = np.where(suspicious_mask_np)[0]
print(f"Suspicious transactions flagged: {suspicious_mask_np.sum():,}")

# Build account-level scores and transaction lists
account_scores       = defaultdict(float)
account_trans_count  = defaultdict(int)
account_transactions = defaultdict(list)

idx2node = graph_data.idx2node if hasattr(graph_data, "idx2node") else {}

src_all = graph_data.edge_index[0].cpu().numpy()
dst_all = graph_data.edge_index[1].cpu().numpy()

for edge_idx in tqdm(suspicious_indices, desc="Building account scores"):
    src_node = idx2node.get(int(src_all[edge_idx]), f"ACC_{src_all[edge_idx]}")
    dst_node = idx2node.get(int(dst_all[edge_idx]), f"ACC_{dst_all[edge_idx]}")
    prob     = float(all_probs[edge_idx])

    account_scores[src_node] += prob
    account_scores[dst_node] += prob
    account_trans_count[src_node] += 1
    account_trans_count[dst_node] += 1

    row = trans_clean.iloc[edge_idx]
    record = {
        "transaction_id": f"T{edge_idx}",
        "sender_id"     : src_node,
        "receiver_id"   : dst_node,
        "amount"        : float(row["Amount Paid"]),
        "timestamp"     : str(row["Timestamp"]),
    }
    if len(account_transactions[src_node]) < 20:
        account_transactions[src_node].append(record)
    if len(account_transactions[dst_node]) < 20:
        account_transactions[dst_node].append(record)

# Normalise to [0, 100]
if account_scores:
    max_score = max(account_scores.values())
    account_scores = {
        k: min(100.0, (v / (max_score + 1e-8)) * 100)
        for k, v in account_scores.items()
    }


def detect_fraud_rings(suspicious_edge_idx, src_all, dst_all, idx2node, account_scores):
    adj = defaultdict(set)
    for eidx in suspicious_edge_idx:
        s = idx2node.get(int(src_all[eidx]), f"ACC_{src_all[eidx]}")
        d = idx2node.get(int(dst_all[eidx]), f"ACC_{dst_all[eidx]}")
        adj[s].add(d)
        adj[d].add(s)

    visited = set()
    rings   = {}
    ring_id = 0

    for start in adj:
        if start in visited:
            continue
        members = []
        queue   = deque([start])
        visited.add(start)
        while queue:
            node = queue.popleft()
            members.append(node)
            for nb in adj[node]:
                if nb not in visited:
                    visited.add(nb)
                    queue.append(nb)
        if len(members) >= 2:
            risk = float(np.mean([account_scores.get(m, 0) for m in members]))
            rings[f"RING_{ring_id:03d}"] = {
                "member_accounts": members,
                "pattern_type"   : "network_cluster",
                "risk_score"     : risk,
            }
            ring_id += 1

    return rings


fraud_rings = detect_fraud_rings(
    suspicious_indices, src_all, dst_all, idx2node, account_scores
)
print(f"Fraud rings detected: {len(fraud_rings):,}")

Suspicious transactions flagged: 6,924,049


Building account scores:   0%|          | 0/6924049 [00:00<?, ?it/s]

In [ ]:
def generate_report(account_scores, account_transactions, fraud_rings,
                    graph_data, cfg, trans_clean):
    t0 = time.time()

    suspicious_accounts = []
    for acc_id, score in account_scores.items():
        if score < cfg.SUSPICION_THRESHOLD * 100:
            continue

        ring_id = next(
            (rid for rid, info in fraud_rings.items()
             if acc_id in info["member_accounts"]),
            None
        )

        # Pattern tags based on graph topology heuristics
        patterns = []
        tx_count = account_trans_count.get(acc_id, 0)
        if tx_count > 5:
            patterns.append("high_velocity")
        if ring_id:
            patterns.append("network_cluster")
        if score > 80:
            patterns.append("high_risk_score")

        suspicious_accounts.append({
            "account_id"       : acc_id,
            "suspicion_score"  : round(score, 2),
            "detected_patterns": patterns,
            "ring_id"          : ring_id,
            "transactions"     : account_transactions.get(acc_id, []),
        })

    suspicious_accounts.sort(key=lambda x: x["suspicion_score"], reverse=True)

    report = {
        "suspicious_accounts": suspicious_accounts,
        "fraud_rings": [
            {
                "ring_id"        : rid,
                "member_accounts": info["member_accounts"],
                "pattern_type"   : info["pattern_type"],
                "risk_score"     : round(info["risk_score"], 2),
            }
            for rid, info in fraud_rings.items()
        ],
        "summary": {
            "total_accounts_analyzed"  : graph_data.num_nodes,
            "suspicious_accounts_flagged": len(suspicious_accounts),
            "fraud_rings_detected"     : len(fraud_rings),
            "processing_time_seconds"  : round(time.time() - t0, 3),
            "dataset"                  : cfg.DATASET_NAME,
            "model"                    : "EdgeGAT",
            "suspicion_threshold"      : cfg.SUSPICION_THRESHOLD,
            "test_f1"                  : round(float(test_f1), 4),
            "test_precision"           : round(float(precision), 4),
            "test_recall"              : round(float(recall), 4),
        },
        "generated_at": datetime.now().isoformat(),
    }
    return report


aml_report = generate_report(
    account_scores, account_transactions, fraud_rings,
    graph_data, cfg, trans_clean
)

print("=" * 60)
print("AML DETECTION REPORT - SUMMARY")
print("=" * 60)
print(json.dumps(aml_report["summary"], indent=2))

print("\nTop 5 Suspicious Accounts:")
for acc in aml_report["suspicious_accounts"][:5]:
    print(f"  {acc['account_id']}  score={acc['suspicion_score']}  "
          f"ring={acc['ring_id']}  patterns={acc['detected_patterns']}")

print("\nTop 5 Fraud Rings:")
for ring in aml_report["fraud_rings"][:5]:
    print(f"  {ring['ring_id']}  members={len(ring['member_accounts'])}  "
          f"risk={ring['risk_score']}  type={ring['pattern_type']}")

In [ ]:
out_path = f"/kaggle/working/aml_report_{cfg.DATASET_NAME}.json"
with open(out_path, "w") as f:
    json.dump(aml_report, f, indent=2, default=str)

print(f"Report saved to: {out_path}")
print("\nFULL JSON REPORT (first 3 suspicious accounts):")
preview = dict(aml_report)
preview["suspicious_accounts"] = aml_report["suspicious_accounts"][:3]
preview["fraud_rings"]         = aml_report["fraud_rings"][:3]
print(json.dumps(preview, indent=2, default=str))

In [19]:
# t Save the Trained Model
import os

# Define the save path
save_path = os.path.join("/kaggle/working/", f"model_{cfg.DATASET_NAME}.pth")

# Save the best_state captured during training
if best_state is not None:
    torch.save(best_state, save_path)
    print(f"Model saved successfully to: {save_path}")
else:
    print("Error: No trained model state found to save.")

Model saved successfully to: /kaggle/working/model_LI-Small.pth
